<a href="https://colab.research.google.com/github/paolobalasso/DataScienceCourse/blob/main/notebooks/01_churn_cost_threshold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 01 · Churn — Beyond the Threshold

**Data Science · Università Cattolica del Sacro Cuore, Milan**
Instructor: *Paolo Balasso*

---

## What this notebook adds to the slides

The churn module of the slides (*Churn Prediction & Cost-Based Decision*) already covered:
- Logistic regression for churn (formula, β interpretation, MLE)
- Feature engineering (behavioural, contractual, geographic)
- Confusion matrix → Accuracy / Precision / Recall / F1
- **Cost matrix and the optimal-threshold formula** τ* = C_FP / (C_FP + C_FN)
- Worked numeric examples and the class churn example at τ = 0.5, then τ*

This notebook fills **four gaps** that did not fit on slides:

| # | Topic | Why it matters |
|---|-------|----------------|
| **G1** | **ROC vs PR curve** | ROC can look excellent while precision at a useful recall is poor. PR shows what each extra churner caught costs you. |
| **G2** | **Calibration** (Platt / isotonic + reliability diagram + Brier) | A model can rank well and still output wrong probabilities. Cost-based decisions need *calibrated* probabilities. |
| **G3** | **Lift & Gain curves + deciles** | The marketing team thinks in *who do we call first*, not in *AUC*. |
| **G4** | **From probability to action, in euros** + a brief note on uplift modelling | VIP retention call, email or no contact — decided customer by customer with one expected-value rule. |

It closes with the **cost-based threshold**: chosen on a validation set, reported once on the test set, with the retention rate *r* as a parameter.

**One economic model, everywhere.** Sections 4, 6 and 7 all use the same business parameters, set once in Section 0.

**Style.** Short, readable functions. Mini-exercises are mostly *parameter tweaks* — change a cost, swap a calibrator, regroup into quintiles — to reinforce the concept, not to test coding skill.

**How to run.** *Runtime → Run all* at the start. The whole notebook takes about one minute on Colab CPU; the calibration cell in Section 4 is the slow one.

---

## 0 · Setup

In [ ]:
# Colab already ships every package used here, so there is nothing to install.
# Running locally? Install once from a terminal:
#   pip install "scikit-learn>=1.3" scipy pandas numpy matplotlib seaborn
import sys, sklearn, scipy
print(f'Python {sys.version.split()[0]} | scikit-learn {sklearn.__version__} | scipy {scipy.__version__}')

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)   # hide library deprecation notices only
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import brentq

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_curve, precision_recall_curve, roc_auc_score, average_precision_score,
    brier_score_loss, confusion_matrix,
)

NAVY, GOLD, INK, MUTED = '#1B263B', '#D4A017', '#0F141E', '#556070'
GREEN, RED = '#2D6E3E', '#B32D2D'
sns.set_style('whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans',
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK})
np.random.seed(42)
print('Setup complete.')

### Business parameters — one economic model for the whole notebook

Contacting a customer costs **C_offer**. If that customer was going to churn, the offer keeps them with probability **r** (the *retention rate*), and we keep their value **CLV**. So the expected net value of contacting a customer with churn probability *p* is

$$EV(p) = p \cdot r \cdot CLV - C_{offer} \qquad\Rightarrow\qquad \text{contact if } p > \frac{C_{offer}}{r \cdot CLV}$$

This is the cost matrix of the slides in another form: **C_FP = C_offer** (an offer wasted on a customer who would have stayed) and **C_FN = r·CLV − C_offer** (the net value we lose on a churner we did not contact). Then C_offer / (r·CLV) is exactly **τ\* = C_FP / (C_FP + C_FN)**.

With the slides' numbers — €30 offer, r = 1, CLV = €530 — we get C_FN = €500 and **τ\* = 30 / 530 = 0.057**. Section 6 reuses the same CLV with two real-world actions (save rates below 100%); Section 7 lets *r* move.

In [ ]:
# ONE economic model, used in Sections 4, 6 and 7
CLV     = 530.0   # value of a customer we keep (EUR)
C_OFFER = 30.0    # cost of the standard retention offer (EUR)
R_OFFER = 1.0     # retention rate r: share of contacted would-be churners the offer keeps

# The same model in the cost-matrix language of the slides
C_FP = C_OFFER                     # offer wasted on a customer who would have stayed
C_FN = R_OFFER * CLV - C_OFFER     # net value lost on a churner we did not contact
TAU_STAR = C_FP / (C_FP + C_FN)    # identical to C_OFFER / (R_OFFER * CLV)


def expected_cost(p, y, tau, c_fp=C_FP, c_fn=C_FN):
    # Cost of the policy 'contact everyone with p >= tau', measured on labelled data
    pred = (p >= tau).astype(int)
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    return c_fp * fp + c_fn * fn, fp, fn


print(f'C_FP = EUR {C_FP:.0f}  |  C_FN = EUR {C_FN:.0f}  |  '
      f'tau* = {C_FP:.0f} / {C_FP + C_FN:.0f} = {TAU_STAR:.3f}')

## 1 · Build a Telco-style synthetic churn dataset

We generate a **synthetic** dataset rather than relying on an external download — so the notebook runs offline and the data-generating process is *transparent*.

The features mirror the Telco-Customer-Churn benchmark structure: tenure, monthly charges, contract type, plus a noisy "competitor coverage gap" feature.

**Class imbalance** is set to **22% churners** (realistic for telecom) — this matters in Section 3 (ROC vs PR). The generator shifts the intercept of the risk score until the *average* churn probability equals the target, so the realised churn rate lands on 22% (printed below).

In [ ]:
CHURN_RATE = 0.22   # target share of churners (the Section 3 mini-exercise changes this)

def make_churn(n=8000, churn_rate=CHURN_RATE, seed=42):
    rng = np.random.default_rng(seed)
    # Latent risk score: tenure -, monthly_charges +, contract long -, competitor_gap +
    tenure         = rng.integers(1, 73, n)                  # months 1..72
    monthly_charge = rng.normal(70, 30, n).clip(20, 200)
    contract       = rng.choice([0, 1, 2], n, p=[0.55, 0.25, 0.20])  # 0=monthly,1=1yr,2=2yr
    competitor_gap = rng.normal(0, 1, n)                     # standardised
    payment_auto   = rng.choice([0, 1], n, p=[0.4, 0.6])     # 1 = auto, lower churn
    support_calls  = rng.poisson(1.5, n)
    # Latent risk
    z = (-0.05*tenure + 0.02*monthly_charge - 0.6*contract
         + 0.45*competitor_gap - 0.35*payment_auto + 0.15*support_calls)
    # Calibrate to the target churn rate: find the intercept b that makes the
    # AVERAGE churn probability equal to churn_rate (brentq is a 1-D root finder)
    b = brentq(lambda b: (1 / (1 + np.exp(-(z + b)))).mean() - churn_rate, -30, 30)
    p = 1 / (1 + np.exp(-(z + b)))
    y = (rng.uniform(size=n) < p).astype(int)
    df = pd.DataFrame({
        'tenure_months':    tenure,
        'monthly_charge':   monthly_charge.round(2),
        'contract_type':    contract,              # 0/1/2
        'competitor_gap':   competitor_gap.round(3),
        'payment_auto':     payment_auto,
        'support_calls':    support_calls,
        'churned':          y,
    })
    return df

df = make_churn()
print(f'Shape: {df.shape}')
print(f'Churn rate: target {CHURN_RATE:.0%}, realised {df["churned"].mean():.2%}')
df.head()

**First look — six commands.** Before any chart, run the same six commands on every new dataset: `shape`, `dtypes`, `isna`, `duplicated`, `describe`, and `value_counts` on the target. On synthetic data they are boring by design (no missing values, no duplicates). On real data, this is where the surprises are.

In [ ]:
print('shape      :', df.shape)
print('duplicates :', int(df.duplicated().sum()))
print('missing    :', int(df.isna().sum().sum()), 'cells')
print('\ndtypes:')
print(df.dtypes.to_string())
print('\ntarget value_counts:')
print(df['churned'].value_counts().to_string())
df.describe().round(2)

In [ ]:
# Quick EDA: feature distributions by churn label
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
plots = [
    ('tenure_months',   'Tenure (months)'),
    ('monthly_charge',  'Monthly charge (€)'),
    ('contract_type',   'Contract type (0=monthly, 2=2yr)'),
    ('competitor_gap',  'Competitor coverage gap (z)'),
    ('payment_auto',    'Auto-payment'),
    ('support_calls',   'Support calls (last 90d)'),
]
for ax, (col, label) in zip(axes.flat, plots):
    for c, color, name in [(0, NAVY, 'Stayed'), (1, RED, 'Churned')]:
        ax.hist(df.loc[df['churned']==c, col], bins=25, alpha=0.55,
                color=color, label=name, edgecolor='white', linewidth=0.3)
    ax.set_title(label, fontweight='bold', color=INK, fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle('Feature distributions by churn label',
             fontweight='bold', color=INK, y=1.02)
plt.tight_layout(); plt.show()

## 2 · Fit two models (recap of *Logistic Regression for Churn*)

We fit **Logistic Regression** (the slide model) and **Random Forest** (a non-linear comparator). We only need their `predict_proba` outputs for everything that follows.

**Three-way split: 60% train / 20% validation / 20% test.** Every choice we make later — the threshold τ, the tier cut-offs — is made on **validation**. The **test** set is used once, to report the result. This is the CRISP-DM rule from the slides: never tune on the test set.

*Note.* `contract_type` (0/1/2) enters the model as a number because our data-generating process is linear in it. On real data you would one-hot encode it, as in the class churn example.

In [ ]:
X = df.drop(columns='churned')
y = df['churned'].values

# 60% train / 20% validation / 20% test, all stratified on the target
X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.20,
                                            stratify=y, random_state=42)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.25,
                                          stratify=y_tmp, random_state=42)

scaler = StandardScaler().fit(X_tr)          # fit on train only
X_tr_s = scaler.transform(X_tr)
X_va_s = scaler.transform(X_va)
X_te_s = scaler.transform(X_te)

logreg = LogisticRegression(max_iter=1000).fit(X_tr_s, y_tr)
rf     = RandomForestClassifier(n_estimators=300, max_depth=8,
                                random_state=42, n_jobs=-1).fit(X_tr, y_tr)

p_log = logreg.predict_proba(X_te_s)[:, 1]   # test-set probabilities
p_rf  = rf.predict_proba(X_te)[:, 1]
p_rf_va = rf.predict_proba(X_va)[:, 1]       # validation-set probabilities

for name, yy in [('Train', y_tr), ('Validation', y_va), ('Test', y_te)]:
    print(f'{name:<11} {len(yy):>5} customers, {yy.mean():.2%} churned')
print(f'\nLogReg AUC (test):  {roc_auc_score(y_te, p_log):.3f}')
print(f'RF AUC (test):      {roc_auc_score(y_te, p_rf):.3f}')

In [ ]:
# Odds ratios: how the odds of churning multiply for +1 standard deviation of each feature
odds = pd.DataFrame({
    'coef (per 1 SD)':  logreg.coef_[0],
    'odds ratio':       np.exp(logreg.coef_[0]),
    '1 SD in units':    X_tr.std().values,
}, index=X.columns).sort_values('odds ratio', ascending=False).round(3)
odds

**Read-out.** An odds ratio above 1 raises churn risk; below 1 lowers it. The signs match the generator in Section 1: more support calls, higher charges and a larger competitor gap push churn up; tenure, longer contracts and auto-payment push it down. Tenure is the strongest protective lever: +1 SD (about 21 months) cuts the odds of churning by about two thirds (odds ratio ≈ 0.34). Monthly charge is the strongest risk driver: +1 SD (about €29) multiplies the odds by about 1.8. This is the interpretability argument of *Logistic Regression for Churn*: one readable number per feature.

## 3 · Gap G1 — ROC vs PR curve under class imbalance

**ROC** plots TPR vs FPR. The FPR denominator is *all* non-churners (78% of customers here), so even many false alarms look like a small FPR. **ROC can look excellent while precision at a useful recall is poor.**

**PR (Precision-Recall)** plots Precision vs Recall. The Precision denominator is the *predicted positives* — exactly the customers you would contact — so PR shows the price, in wasted contacts, of each extra churner caught.

Two reading rules:
- A useless model has ROC AUC = 0.5 at *any* prevalence.
- On PR, the useless model is the horizontal line at the prevalence. **Average precision (AP) must always be read against that line.**

Rule of thumb in industry: when the positive class is small (say below 30%), always look at PR alongside ROC.

In [ ]:
def plot_roc_pr(y_true, probs_dict):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for name, (p, color) in probs_dict.items():
        # ROC
        fpr, tpr, _ = roc_curve(y_true, p)
        auc = roc_auc_score(y_true, p)
        axes[0].plot(fpr, tpr, color=color, lw=1.6, label=f'{name}  AUC={auc:.3f}')
        # PR
        prec, rec, _ = precision_recall_curve(y_true, p)
        ap = average_precision_score(y_true, p)
        axes[1].plot(rec, prec, color=color, lw=1.6, label=f'{name}  AP={ap:.3f}')
    axes[0].plot([0,1],[0,1], ls='--', color=MUTED, lw=0.8)
    axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC curve', fontweight='bold', color=INK)
    axes[0].legend(loc='lower right')

    baseline = y_true.mean()
    axes[1].axhline(baseline, ls='--', color=MUTED, lw=0.8,
                     label=f'Random ({baseline:.0%})')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall curve', fontweight='bold', color=INK)
    axes[1].legend(loc='lower left')
    plt.tight_layout(); plt.show()


def precision_at_recall(y_true, p, target):
    # Best precision we can get while still catching at least `target` of the churners
    prec, rec, _ = precision_recall_curve(y_true, p)
    return prec[rec >= target].max()


plot_roc_pr(y_te, {
    'LogReg': (p_log, NAVY),
    'Random Forest': (p_rf, GOLD),
})

pd.DataFrame({
    name: {'ROC AUC':                 roc_auc_score(y_te, p),
           'AP (area under PR)':      average_precision_score(y_te, p),
           'AP / prevalence':         average_precision_score(y_te, p) / y_te.mean(),
           'precision @ recall 50%':  precision_at_recall(y_te, p, 0.5),
           'precision @ recall 80%':  precision_at_recall(y_te, p, 0.8)}
    for name, p in [('LogReg', p_log), ('Random Forest', p_rf)]
}).round(3)

**Read-out.** The two models are practically equivalent here (AUC ≈ 0.81 vs 0.80, AP ≈ 0.57 vs 0.55). The PR curve adds what ROC cannot show: what precision you pay for each extra churner caught. To catch half of the churners, a bit more than 5 in 10 contacted customers are real churners; to catch 80%, only 4 in 10 are. The PR baseline is the prevalence (22%) — an AP of about 0.56 is roughly 2.5× that line.

> **Mini-exercise (parametric, no coding).** In Section 1 set `CHURN_RATE = 0.05` — a far more imbalanced case — and *Run all*. Watch the LogReg numbers: ROC AUC moves only a little (about 0.81 → 0.78) while AP collapses (about 0.57 → 0.22). Ranking quality barely changes; precision is simply much harder to get when churners are rare — *that* is what ROC hides. Read AP against its new baseline (5%): LogReg's AP is still about 4–5× the random line. (Set it back to 0.22 afterwards. With only about 80 churners in the test set, expect noisy curves; the Random Forest loses more than LogReg here.)

## 4 · Gap G2 — Calibration: probabilities you can trust

A model can **rank well** (high AUC) and still output probabilities that are wrong *as probabilities*. Three typical cases:
- **Logistic regression trained on the natural class mix** is usually close to calibrated.
- **The same logistic regression trained with `class_weight='balanced'`** (or on a re-balanced 50/50 sample — a very common recipe for imbalanced data) keeps the ranking but **inflates every probability**: it behaves as if churn were 50/50.
- **Random Forest** averages many trees, which can pull probabilities towards the middle and compress the extremes. How much depends on the data — check it, do not assume it.

Why this matters: the cost-based threshold τ\* = C_FP / (C_FP + C_FN) assumes the probability is a *real* probability. If your model says 0.30 but the actual frequency at that score is 0.10, you over-act.

Two standard fixes:
- **Platt scaling** (sigmoid): fits a small logistic curve on the scores
- **Isotonic regression**: non-parametric monotone fit, more flexible but data-hungry

Metric: **Brier score** = mean squared error between predicted probability and outcome (lower = better). Read it against the **no-information Brier** = prevalence × (1 − prevalence), the score of predicting the churn rate for everyone. **Brier skill** = 1 − Brier / no-information Brier (0 = no better than the base rate, 1 = perfect).

In [ ]:
# 1) A typical mis-calibrated model: the same logistic regression, trained with class re-weighting
logreg_bal = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_tr_s, y_tr)
#    ... and its calibrated version (Platt scaling, 5-fold CV on the training set)
logreg_bal_cal = CalibratedClassifierCV(
    LogisticRegression(max_iter=1000, class_weight='balanced'),
    method='sigmoid', cv=5).fit(X_tr_s, y_tr)

# 2) Random Forest calibrated with both methods (5-fold CV) -- the slow part of the notebook
rf_platt    = CalibratedClassifierCV(RandomForestClassifier(n_estimators=300, max_depth=8,
                                                             random_state=42, n_jobs=-1),
                                      method='sigmoid', cv=5).fit(X_tr, y_tr)
rf_isotonic = CalibratedClassifierCV(RandomForestClassifier(n_estimators=300, max_depth=8,
                                                             random_state=42, n_jobs=-1),
                                      method='isotonic', cv=5).fit(X_tr, y_tr)

# Test-set probabilities
p_bal     = logreg_bal.predict_proba(X_te_s)[:, 1]
p_bal_cal = logreg_bal_cal.predict_proba(X_te_s)[:, 1]
p_rf_pl   = rf_platt.predict_proba(X_te)[:, 1]
p_rf_is   = rf_isotonic.predict_proba(X_te)[:, 1]
# Validation-set probabilities (used in Sections 6 and 7 to make choices)
p_bal_va   = logreg_bal.predict_proba(X_va_s)[:, 1]
p_rf_is_va = rf_isotonic.predict_proba(X_va)[:, 1]

preds = {
    'LogReg':                   p_log,
    'LogReg balanced (raw)':    p_bal,
    'LogReg balanced + Platt':  p_bal_cal,
    'RF (raw)':                 p_rf,
    'RF + Platt':               p_rf_pl,
    'RF + Isotonic':            p_rf_is,
}
brier_ref = y_te.mean() * (1 - y_te.mean())     # no-information Brier
print(f'Observed churn rate (test): {y_te.mean():.3f}   |   no-information Brier: {brier_ref:.4f}')

scores = pd.DataFrame({
    'AUC':              [roc_auc_score(y_te, p) for p in preds.values()],
    'Brier':            [brier_score_loss(y_te, p) for p in preds.values()],
    'Brier skill':      [1 - brier_score_loss(y_te, p) / brier_ref for p in preds.values()],
    'mean predicted p': [p.mean() for p in preds.values()],
}, index=list(preds)).round(4)
scores

In [ ]:
# Reliability diagram: predicted probability bins vs observed frequency
N_BINS_CAL = 10
fig, ax = plt.subplots(figsize=(7, 6))
for name, color, ls, marker in [
    ('LogReg',                  NAVY, '-',  'o'),
    ('LogReg balanced (raw)',   RED,  '-',  'o'),
    ('LogReg balanced + Platt', RED,  '--', 's'),
    ('RF (raw)',                GOLD, '-',  'o'),
    ('RF + Isotonic',           GOLD, '--', 's'),
]:
    frac_pos, mean_pred = calibration_curve(y_te, preds[name], n_bins=N_BINS_CAL,
                                            strategy='quantile')
    ax.plot(mean_pred, frac_pos, marker=marker, ls=ls, lw=1.5, color=color, label=name)
ax.plot([0,1],[0,1], ls='--', color=MUTED, lw=0.8, label='Perfect calibration')
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Observed frequency')
ax.set_title(f'Reliability diagram ({N_BINS_CAL} quantile bins)',
             fontweight='bold', color=INK)
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

**Read-out.** Three different stories on one chart.
- **LogReg** sits on the diagonal and has the best Brier score (tied with the Platt-calibrated balanced model): trained on the natural class mix, it is close to calibrated by construction.
- **LogReg balanced** has *exactly the same AUC* — the ranking is identical — but its curve lies far **below** the diagonal (over-confident): when it says about 0.64, only about a third of those customers churn. Its mean predicted probability is about 0.41 against a real churn rate of 0.22, and its Brier score (≈ 0.177) is *worse* than the no-information Brier (≈ 0.171) — negative skill. Platt scaling brings it back to the diagonal at no cost in AUC.
- **Random Forest (raw)** is already close to the diagonal here: slightly over-confident in the low-middle bins, slightly *under*-confident in the top bin. Isotonic calibration changes little (Brier ≈ 0.133 → 0.132). Bagged trees are often only mildly mis-calibrated (Niculescu-Mizil & Caruana, 2005) — the big calibration errors in practice come from re-balancing tricks like the one above.

Lesson: re-balancing tricks change the probabilities, not the ranking — always calibrate (or at least check the reliability diagram) before you apply a cost-based threshold.

> **Mini-exercise (parametric).** Change `N_BINS_CAL = 10` to `5` and to `20`. Few bins = smoother but less granular; many bins = noisier. There is no "right" answer — there is a *trade-off* to internalise.

**What mis-calibration costs in euros.** Apply the *same* threshold from the slides, τ\* = 0.057, to the probabilities of each model (test set):

In [ ]:
rows = []
for name in ['LogReg', 'LogReg balanced (raw)', 'LogReg balanced + Platt', 'RF + Isotonic']:
    p = preds[name]
    cost, fp, fn = expected_cost(p, y_te, TAU_STAR)
    rows.append({'model': name, 'contacted %': 100 * (p >= TAU_STAR).mean(),
                 'FP': fp, 'FN': fn, 'cost (EUR)': cost})
pd.DataFrame(rows).set_index('model').round(1)

**Read-out.** With inflated probabilities, the balanced model pushes almost everyone over τ\*: it contacts about 95% of customers (against 75–82% for the calibrated models) and it is the most expensive line in the table, about €3.7k more than the calibrated LogReg on 1,600 customers. Same ranking, same formula — only the probabilities are wrong. After Platt scaling it behaves like the other calibrated models. The closed-form τ\* is only as good as the probabilities you feed it. (With rarer churn the gap widens sharply: try it with `CHURN_RATE = 0.05`.)

## 5 · Gap G3 — Lift & Gain curves + deciles

Marketing teams do not think "AUC". They think **"if we call the top 10% most-likely-to-churn customers, what fraction of actual churners do we catch?"**

- **Gain curve** answers exactly that: cumulative % of churners captured as you walk down the sorted list.
- **Lift curve** = churn rate inside each bin / overall churn rate. "Lift = 3 in the top decile" means those customers churn 3× more often than average, so calling them is 3× better than random calling. (Cumulative lift = gain / share contacted; for the top bin the two are the same.)

Used everywhere in CRM, retention, and direct marketing campaigns.

In [ ]:
NB = 10   # number of bins: 10 = deciles (the mini-exercise changes it)

def gain_lift_table(y_true, p, n_bins=NB):
    # Sort customers by predicted prob (desc), split into n_bins equal groups, compute gain & lift
    s = pd.DataFrame({'y': y_true, 'p': p}).sort_values('p', ascending=False).reset_index(drop=True)
    s['decile'] = pd.qcut(s.index, n_bins, labels=False) + 1
    grp = s.groupby('decile').agg(
        n=('y', 'size'),
        churners=('y', 'sum'),
        avg_prob=('p', 'mean'),
    )
    total_churners = grp['churners'].sum()
    grp['cum_churners'] = grp['churners'].cumsum()
    grp['gain']         = grp['cum_churners'] / total_churners
    grp['churn_rate']   = grp['churners'] / grp['n']
    grp['lift']         = grp['churn_rate'] / y_true.mean()
    return grp.round(3)

gl_log = gain_lift_table(y_te, p_log, n_bins=NB)
gl_rf  = gain_lift_table(y_te, p_rf_is, n_bins=NB)

print(f'LogReg — gain & lift by bin (bin 1 = top {100/NB:.0f}% riskiest):')
display(gl_log[['n','churners','avg_prob','churn_rate','gain','lift']])

In [ ]:
# Plot gain + lift side-by-side; x = share of customers contacted, so any NB is comparable
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
share = np.arange(1, len(gl_log) + 1) / len(gl_log)       # 0.1, 0.2, ... for deciles

# Gain (both curves start at the origin: contact nobody, catch nobody)
x0 = np.r_[0, share]
axes[0].plot([0, 1], [0, 1], ls='--', color=MUTED, lw=0.9, label='Random')
axes[0].plot(x0, np.r_[0, gl_log['gain'].values], marker='o', color=NAVY, lw=1.6, label='LogReg')
axes[0].plot(x0, np.r_[0, gl_rf['gain'].values],  marker='s', color=GOLD, lw=1.6, label='RF + Iso')
axes[0].set_xlabel('Share of customers contacted (riskiest first)')
axes[0].set_ylabel('Cumulative % churners captured')
axes[0].set_title('Gain curve', fontweight='bold', color=INK)
axes[0].legend(loc='lower right')

# Lift
axes[1].axhline(1.0, ls='--', color=MUTED, lw=0.9, label='Random = 1.0×')
axes[1].plot(share, gl_log['lift'].values, marker='o', color=NAVY, lw=1.6, label='LogReg')
axes[1].plot(share, gl_rf['lift'].values,  marker='s', color=GOLD, lw=1.6, label='RF + Iso')
axes[1].set_xlabel('Share of customers contacted (bin)'); axes[1].set_ylabel('Lift (vs random)')
axes[1].set_title('Lift curve', fontweight='bold', color=INK)
axes[1].legend(loc='upper right')
plt.tight_layout(); plt.show()

print(f'\nTop-bin lift  (LogReg): {gl_log.loc[1, "lift"]:.2f}x')
print(f'Top-bin gain  (LogReg): {gl_log.loc[1, "gain"]:.1%} of churners captured '
      f'in the top {100/NB:.0f}% of customers')

> **Mini-exercise (parametric).** Switch `NB = 10` (deciles) to `NB = 5` (quintiles) or `NB = 20`. Note how the *shape* of the gain curve does not change — only the resolution — and how the top-bucket lift typically increases as bins shrink (concentration effect). This is *not* a model improvement; it is a binning artefact. A frequent trap in client pitches.

## 6 · Gap G4 — From probabilities to action tiers, in euros

From a probability to a business action. The retention team has two plays, plus doing nothing:

| Action | Unit cost | Save rate *r* (share of contacted would-be churners kept) |
|--------|-----------|------------------------------------------------------------|
| **VIP retention call** — human agent, custom offer | €100 | 30% |
| **Targeted email** — personalised email + 1-month discount | €5 | 10% |
| **No contact** | €0 | — |

**The expected-value rule.** Same economic model as in Section 0 (CLV = €530). For each customer and each action *a*:

$$EV_a(p) = p \cdot r_a \cdot CLV - cost_a$$

Pick the action with the highest EV; if every EV is negative, do not contact. Each action has its own break-even probability, **p\*_a = cost_a / (r_a · CLV)** — the τ\* formula again, one per action. These are *probability* thresholds, so this is where calibration really matters.

**Baseline — what marketing usually does.** Call the top 10%, email the next 20%, ignore the bottom 70% (quantile tiers). This uses only the *ranking*, so calibration cannot change it, and it ignores the economics. We compare both rules on the test set; the quantile cut-offs are set on validation.

In [ ]:
ACTIONS = {                     # cost in EUR, save = retention rate r of the action
    'VIP':       {'cost': 100.0, 'save': 0.30},
    'Email':     {'cost':   5.0, 'save': 0.10},
    'NoContact': {'cost':   0.0, 'save': 0.00},
}

def ev_tiers(p, actions=ACTIONS, clv=CLV):
    # Per-customer expected-value rule: the action with the highest EV(p) = p*save*CLV - cost
    names = list(actions)
    ev = np.column_stack([p * actions[a]['save'] * clv - actions[a]['cost'] for a in names])
    return np.array(names)[ev.argmax(axis=1)]          # NoContact has EV = 0

def quantile_tiers(p, p_ref, q=(0.70, 0.90)):
    # Baseline: tiers from population quantiles of p_ref (validation set). Ranking only.
    q_low, q_high = np.quantile(p_ref, q)
    tier = np.where(p >= q_high, 'VIP', np.where(p >= q_low, 'Email', 'NoContact'))
    return tier, q_low, q_high

# Break-even probabilities: the per-action tau*
for a in ['VIP', 'Email']:
    c, r = ACTIONS[a]['cost'], ACTIONS[a]['save']
    print(f'{a:<6} worth it when p > {c:.0f} / ({r:.2f} x {CLV:.0f}) = {c / (r * CLV):.3f}')
v, e = ACTIONS['VIP'], ACTIONS['Email']
if v['save'] > e['save']:
    p_vip = (v['cost'] - e['cost']) / ((v['save'] - e['save']) * CLV)
    print(f'VIP beats Email only when p > {p_vip:.3f}')
elif v['cost'] >= e['cost']:
    print('VIP never beats Email: it costs at least as much and keeps no more churners')
else:
    print('VIP always beats Email: it is cheaper and keeps at least as many churners')

tier_q, q_low, q_high = quantile_tiers(p_rf_is, p_rf_is_va)
print(f'\nBaseline quantile cuts (from validation): Email if p >= {q_low:.3f}, VIP if p >= {q_high:.3f}')

In [ ]:
# Business simulation on the test set (back-test on realised labels):
# each contacted would-be churner is kept with probability save_a
def campaign_value(tier, y, actions=ACTIONS, clv=CLV):
    rows = []
    for a in actions:
        m = tier == a
        n, churners = int(m.sum()), int(y[m].sum())
        cost  = n * actions[a]['cost']
        saved = churners * actions[a]['save']
        rows.append({'Tier': a, 'N': n, 'Churners would-be': churners,
                     'Saved (estimate)': saved, 'Cost': cost,
                     'Gross value': saved * clv, 'Net value': saved * clv - cost})
    out = pd.DataFrame(rows).set_index('Tier')
    out.loc['TOTAL'] = out.sum()
    return out.astype({'N': int, 'Churners would-be': int, 'Cost': int}).round(1)

print('BASELINE - quantile tiers (top 10% VIP, next 20% Email), RF + Isotonic:')
display(campaign_value(tier_q, y_te))
print('EXPECTED-VALUE RULE, RF + Isotonic:')
display(campaign_value(ev_tiers(p_rf_is), y_te))

In [ ]:
# Does calibration matter? Same two rules, different probabilities (test set)
def summary(tier):
    v = campaign_value(tier, y_te)
    return {'VIP': int(v.loc['VIP', 'N']), 'Email': int(v.loc['Email', 'N']),
            'No contact': int(v.loc['NoContact', 'N']), 'Net value (EUR)': v.loc['TOTAL', 'Net value']}

policies = {
    'Quantile tiers - RF + Isotonic (baseline)': quantile_tiers(p_rf_is, p_rf_is_va)[0],
    'Quantile tiers - RF raw':                   quantile_tiers(p_rf,    p_rf_va)[0],
    'EV rule - RF + Isotonic':                   ev_tiers(p_rf_is),
    'EV rule - RF raw':                          ev_tiers(p_rf),
    'EV rule - LogReg':                          ev_tiers(p_log),
    'EV rule - LogReg balanced (raw)':           ev_tiers(p_bal),
    'EV rule - LogReg balanced + Platt':         ev_tiers(p_bal_cal),
}
pd.DataFrame({k: summary(t) for k, t in policies.items()}).T.astype(
    {'VIP': int, 'Email': int, 'No contact': int, 'Net value (EUR)': int})

**Read-out.**
- At these save rates the **VIP call almost never pays**: it breaks even only above p ≈ 0.63, and it beats a €5 email only above p ≈ 0.90 — a handful of customers. The quantile baseline nevertheless calls about 10% of the base. Those calls still return a little, but an email to the same customers would return almost four times more.
- The **expected-value rule** emails everyone above p ≈ 0.09 (about two thirds of the base), calls almost nobody, and nearly doubles the net value of the baseline on the same test set (about €11.9k vs €6.2k).
- **Calibration matters for the EV rule, not for the quantile rule.** Quantile tiers on raw and calibrated RF put about 96% of customers in the same tier, because only the ranking counts. The EV rule compares probabilities with euro thresholds: fed with the inflated probabilities of the balanced LogReg, it makes 40 VIP calls and emails 86% of the base, and earns about €1.4k less than the same model after Platt scaling. (Raw RF is nearly calibrated here, so for RF the two lines are close.)

> **Mini-exercise (parametric).** (1) Find the policy that maximises Net Value — is the VIP call worth it at these save rates? Try the baseline with `q=(0.60, 0.85)` and `q=(0.80, 0.95)` in `quantile_tiers`, then any pair you like. Watch how much the cut-offs move the result, and compare your best pair with the EV rule, which needed no tuning at all. (2) Raise the VIP save rate in `ACTIONS` from 0.30 to 0.50, then 0.70. At which save rate does the VIP call start to win customers from the email, and which ones? Check your answer with the printed rule *VIP beats Email when p > (100 − 5) / ((r_VIP − 0.10) × 530)*.

> **Note.** The save rates (0.30 / 0.10) are educated guesses. In practice they come from an **A/B test or uplift model** (Section 8).

## 7 · Recap from slides — cost-based threshold, chosen on validation

The slides (*Cost-Based Decision Matrix*, *Optimal Threshold for Cost Minimization*, *Cost-Based Threshold Optimisation*) derived τ\* = C_FP / (C_FP + C_FN). In this notebook it is the single-action case of the model in Section 0:

- C_FP = C_offer = cost of contacting a non-churner (wasted offer + brand fatigue)
- C_FN = r·CLV − C_offer = net value lost on a churner we did not contact
- so τ\* = C_offer / (r·CLV); with €30, r = 1, CLV = €530 → **τ\* = 30 / 530 = 0.057**

**Procedure.** Sweep τ on the **validation** set and take the empirical minimum. Compare it with the closed form. Then report, once, on the **test** set: the cost at the default τ = 0.5, at τ\*_closed and at τ\*_empirical. Let us turn the knob across several cost pairs.

In [ ]:
P_VA, P_TE = p_rf_is_va, p_rf_is        # calibrated RF: validation to choose, test to report
taus = np.round(np.linspace(0.01, 0.99, 99), 2)
cost_pairs = [(30, 500), (50, 500), (75, 500), (100, 500), (30, 1000)]

fig, ax = plt.subplots(figsize=(11, 5))
rows = []
for c_fp, c_fn in cost_pairs:
    costs_va = [expected_cost(P_VA, y_va, t, c_fp, c_fn)[0] for t in taus]
    tau_closed = c_fp / (c_fp + c_fn)
    tau_emp    = taus[int(np.argmin(costs_va))]      # chosen on VALIDATION
    ax.plot(taus, costs_va, lw=1.4,
            label=f'C_FP={c_fp}, C_FN={c_fn} → τ*_closed={tau_closed:.3f}, τ*_emp={tau_emp:.2f}')
    cost_05   = expected_cost(P_TE, y_te, 0.5, c_fp, c_fn)[0]       # reported on TEST
    cost_star = expected_cost(P_TE, y_te, tau_closed, c_fp, c_fn)[0]
    cost_emp  = expected_cost(P_TE, y_te, tau_emp, c_fp, c_fn)[0]
    rows.append({'C_FP': c_fp, 'C_FN': c_fn,
                 'tau*_closed': round(tau_closed, 3), 'tau*_emp (valid)': tau_emp,
                 'test cost @ 0.5': cost_05, 'test cost @ tau*_closed': cost_star,
                 'test cost @ tau*_emp': cost_emp,
                 'saved vs 0.5': cost_05 - cost_star,
                 'saved %': round(100 * (cost_05 - cost_star) / cost_05, 1),
                 'contacted % @ tau*': round(100 * (P_TE >= tau_closed).mean(), 1)})
ax.axvline(0.5, ls=':', color=MUTED, lw=0.9)
ax.text(0.505, ax.get_ylim()[1] * 0.95, 'default τ = 0.5', color=MUTED, fontsize=8, va='top')
ax.set_xlabel('Threshold τ'); ax.set_ylabel('Expected cost on validation (€)')
ax.set_title('Cost-vs-threshold sweep across cost pairs (validation set)',
             fontweight='bold', color=INK)
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

sweep_table = pd.DataFrame(rows)
sweep_table

In [ ]:
# The slides' cost pair: confusion matrices on the test set at the default 0.5 and at tau*
for tau in [0.5, TAU_STAR]:
    cm = confusion_matrix(y_te, (P_TE >= tau).astype(int))
    cost = expected_cost(P_TE, y_te, tau)[0]
    print(f'tau = {tau:.3f}  ->  cost EUR {cost:,.0f}   (C_FP = {C_FP:.0f}, C_FN = {C_FN:.0f})')
    display(pd.DataFrame(cm, index=['Actual: stay', 'Actual: churn'],
                         columns=['Predicted: stay', 'Predicted: churn']))

**Read-out.** The money is in **leaving 0.5**, not in the third decimal. With the slides' costs (30, 500), moving from τ = 0.5 to τ\* = 0.057 cuts the test-set cost by about 70% (≈ €114.5k → €34.3k on 1,600 customers): false negatives fall from 225 to 10, while false positives rise from 68 to 978 — cheap mistakes traded for expensive ones. The cost curve is **flat around τ\***: the empirical optimum chosen on validation (0.04) and the closed form (0.057) land within about 2% of each other on the test cost. The slides' pair (30, 500) gives the τ\* ≈ 6% of the class check-in; the (75, 500) pair gives τ\* = 0.13: a dearer offer raises the threshold.

The closed form needs no tuning at all — but it is only as good as the calibration of the probabilities (Section 4).

*Note.* The euro figures on the slides *Cost-Based Threshold Optimisation* and *Scaling Impact to 100 000 Customers* come from a different 1,000-customer test set, so they do not match these numbers exactly. The logic is the same.

> **Mini-exercise (parametric).** Add `(20, 800)` to `cost_pairs` and re-run. Then set `P_VA, P_TE = p_bal_va, p_bal` (the mis-calibrated balanced LogReg): the empirical τ\* moves far away from the closed form, and the cost at τ\*_closed jumps. The formula is right only if the probabilities are. (Choosing τ on validation partly rescues a mis-calibrated model; the closed form cannot.)

**What if the offer keeps only a share *r* of the churners we contact?** The slides' τ\* assumes r = 1. With r < 1, C_FN shrinks to r·CLV − C_offer, and the threshold rises: **τ\*(r) = C_offer / (r·CLV)**. Same model, one more knob.

In [ ]:
rows = []
for r in [1.0, 0.8, 0.6, 0.4, 0.3]:
    c_fn_r = r * CLV - C_OFFER                       # net value lost on a missed churner
    tau_r  = C_OFFER / (r * CLV)                     # = C_FP / (C_FP + c_fn_r)
    cost_r  = expected_cost(P_TE, y_te, tau_r, C_OFFER, c_fn_r)[0]
    cost_05 = expected_cost(P_TE, y_te, 0.5,   C_OFFER, c_fn_r)[0]
    rows.append({'r': r, 'C_FN = r*CLV - C_offer': c_fn_r, 'tau*(r)': round(tau_r, 3),
                 'contacted %': round(100 * (P_TE >= tau_r).mean(), 1),
                 'test cost @ tau*(r)': cost_r, 'test cost @ 0.5': cost_05})
pd.DataFrame(rows).set_index('r')

**Read-out.** r = 1 reproduces the slides: τ\* = 0.057. With a more realistic r = 0.6, τ\*(0.6) = 30 / (0.6 × 530) ≈ 0.094 — still far below 0.5, so the main lesson holds. The less effective the offer, the more selective you should be. Read the costs across a row, not down the column: they fall as *r* falls only because each missed churner is worth less (C_FN shrinks). Across a row, τ\*(r) beats 0.5 every time here, but the gap narrows as the offer gets weaker (about €115k → €34k at r = 1, about €31k → €22k at r = 0.3). Section 6 is the same model with two actions, each with its own *r*.

## 8 · Brief note — Uplift modelling (one paragraph)

The pipeline so far predicts *who will churn*. It does **not** predict *who will be saved by a retention offer*. Those are different populations:

- **Persuadable** — would churn without offer, stays with offer → contact them
- **Sure thing** — stays anyway → wasted contact
- **Lost cause** — churns anyway → wasted contact
- **Do-not-disturb** — would stay, but contacting them actually annoys them into churning → avoid!

**Uplift modelling** estimates the *incremental* effect of the offer, not the absolute probability. In the language of Section 6, it estimates the save rate *r* customer by customer instead of assuming one number per action. It needs an experimental control group. Tools: Two-Model approach, Class Transformation, Causal Trees and Causal Forests.

This is the natural next step *after* a calibrated churn model — out of scope for today, but the right mental model for production retention systems.

References: Radcliffe & Surry (2011); Athey & Imbens (2016) for causal trees; Wager & Athey (2018) for causal forests.

## 9 · Caveats & next steps

1. **Synthetic data optimism.** Real telco data has missing values, label noise, leakage, drift. Always inspect.
2. **Calibration drifts.** A model calibrated on Q1 may decalibrate by Q3. Monitor Brier score in production.
3. **Top-decile lift overstated** when the test set is too small. Always bootstrap confidence intervals.
4. **Cost matrix is not constant.** C_FN (lost CLV) depends on customer segment; a single τ\* across segments is sub-optimal.
5. **Choose on validation, report on test.** Thresholds, tier cut-offs and save-rate assumptions are decisions; never tune them on the test set.
6. **Uplift > propensity** for the retention question. Build A/B test infrastructure first.

### Where to go next

- Replace Random Forest with `HistGradientBoostingClassifier` and check its reliability diagram before and after calibration
- Bootstrap top-decile lift to put error bars on it
- Build a segment-aware threshold (one τ\* per contract type)
- Add a third action to `ACTIONS` (for example a €30 offer with a 40% save rate) and see which customers it takes

### References

- Niculescu-Mizil, A. & Caruana, R. (2005). *Predicting good probabilities with supervised learning*. ICML.
- Platt, J. (1999). *Probabilistic outputs for support vector machines and comparisons to regularized likelihood methods*.
- Radcliffe, N. & Surry, P. (2011). *Real-world uplift modelling with significance-based uplift trees*.
- Athey, S. & Imbens, G. (2016). *Recursive partitioning for heterogeneous causal effects*. PNAS.
- Wager, S. & Athey, S. (2018). *Estimation and inference of heterogeneous treatment effects using random forests*. JASA.
- Course slides, churn module (*Churn Prediction & Cost-Based Decision*) and the class churn example (*Cost-Based Threshold Optimisation*) — cost matrix and τ\* derivation.

---

**End of notebook.** About one minute of runtime on Colab CPU. The goal was to close the four gaps the slides could not cover, with parametric mini-exercises to reinforce the concepts.